# Part 3: Data Analytics

This notebook performs data analysis on:
1. BLS time-series data (from Part 1)
2. Population data from DataUSA API (from Part 2)

## Analysis Tasks

1. Calculate mean and standard deviation of US population (2013-2018)
2. Find best year per series_id (year with max sum of values)
3. Generate combined report for PRS30006032 Q01 with population data


## Setup and Imports


In [ ]:
import sys
from pathlib import Path

# Add src to path to import rearc package
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import boto3
import json
from io import StringIO
import numpy as np

# Import analytics queries
from rearc.analytics import (
    query1_population_stats,
    query2_best_year_per_series,
    query3_combined_report
)


## Configuration


In [ ]:
# Configuration
# Option 1: Load from S3 (set USE_LOCAL_DATA = False)
# Option 2: Load from local files (set USE_LOCAL_DATA = True)
USE_LOCAL_DATA = False  # Set to True to use local test data

if USE_LOCAL_DATA:
    # Local data paths (for testing without AWS)
    LOCAL_DATA_DIR = project_root / 'data' / 'local'
    BLS_DATA_PATH = LOCAL_DATA_DIR / 'pr.data.0.Current'
    POPULATION_DATA_PATH = LOCAL_DATA_DIR / 'population_data.json'
    print("Using local data files")
else:
    # S3 Configuration
    BUCKET_NAME = 'your-bucket-name'  # Update with your bucket name
    BLS_DATA_KEY = 'pr.data.0.Current'  # BLS time-series data
    POPULATION_DATA_KEY = 'population_data_latest.json'  # Population data
    
    # Initialize S3 client
    s3_client = boto3.client('s3')
    print("Using S3 data")


## Load Data


In [ ]:
# Load BLS time-series data
if USE_LOCAL_DATA:
    # Load from local file
    bls_content = BLS_DATA_PATH.read_text()
    bls_df = pd.read_csv(StringIO(bls_content), sep='\t')
else:
    # Load from S3
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=BLS_DATA_KEY)
    bls_content = response['Body'].read().decode('utf-8')
    bls_df = pd.read_csv(StringIO(bls_content), sep='\t')

# Clean column names (remove extra spaces)
bls_df.columns = bls_df.columns.str.strip()

print(f"BLS Data Shape: {bls_df.shape}")
print(f"BLS Data Columns: {bls_df.columns.tolist()}")
print(f"\nFirst few rows:")
bls_df.head()


In [ ]:
# Load Population data
if USE_LOCAL_DATA:
    # Load from local file
    population_data = json.loads(POPULATION_DATA_PATH.read_text())
else:
    # Load from S3
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=POPULATION_DATA_KEY)
    population_content = response['Body'].read().decode('utf-8')
    population_data = json.loads(population_content)

# Convert to DataFrame
population_df = pd.DataFrame(population_data.get('data', []))

print(f"Population Data Shape: {population_df.shape}")
print(f"Population Data Columns: {population_df.columns.tolist()}")
print(f"\nYears available: {sorted(population_df['Year'].unique())}")
print(f"\nFirst few rows:")
population_df.head(10)


## Data Cleaning


In [ ]:
# Clean BLS data - trim whitespaces from string columns
string_columns = bls_df.select_dtypes(include=['object']).columns
for col in string_columns:
    if col != 'footnote_codes':  # Skip footnote_codes which may have NaN
        bls_df[col] = bls_df[col].str.strip()

# Ensure proper data types
if 'year' in bls_df.columns:
    bls_df['year'] = pd.to_numeric(bls_df['year'], errors='coerce')
if 'value' in bls_df.columns:
    bls_df['value'] = pd.to_numeric(bls_df['value'], errors='coerce')

print("✓ BLS data cleaned")
print(f"  - Trimmed column names and string values")
print(f"  - Converted year and value to numeric")
print(f"  - Total records: {len(bls_df)}")
print(f"  - Unique series_ids: {bls_df['series_id'].nunique()}")
print(f"  - Year range: {bls_df['year'].min()} - {bls_df['year'].max()}")


## Query 1: Population Statistics (2013-2018)


In [ ]:
# Query 1: Population Statistics (2013-2018)
# Use the analytics module function
result = query1_population_stats(population_df)

print("=" * 60)
print("Query 1: Population Statistics (2013-2018)")
print("=" * 60)
print(f"\nMean Population: {result['mean']:,.0f}")
print(f"Standard Deviation: {result['std_dev']:,.0f}")
print(f"\nFull Result:")
print(result)

# Display the filtered data used for calculation
filtered_pop = population_df[
    (population_df['Year'] >= 2013) & (population_df['Year'] <= 2018)
].sort_values('Year')
print(f"\nData used for calculation ({len(filtered_pop)} years):")
print(filtered_pop[['Year', 'Population']].to_string(index=False))


## Query 2: Best Year per Series ID


In [ ]:
# Query 2: Best Year per Series ID
# Use the analytics module function
result_df = query2_best_year_per_series(bls_df)

print("=" * 60)
print("Query 2: Best Year per Series ID")
print("=" * 60)
print(f"\nTotal series analyzed: {len(result_df)}")
print(f"\nFirst 20 results:")
print(result_df.head(20).to_string(index=False))

# Show some statistics
print(f"\nStatistics:")
print(f"  - Year range in results: {result_df['year'].min()} - {result_df['year'].max()}")
print(f"  - Average best year value: {result_df['value'].mean():.2f}")
print(f"  - Max best year value: {result_df['value'].max():.2f}")

# Show example for PRS30006032
prs32_result = result_df[result_df['series_id'] == 'PRS30006032']
if len(prs32_result) > 0:
    print(f"\nExample - PRS30006032:")
    print(f"  Best year: {prs32_result.iloc[0]['year']}")
    print(f"  Sum value: {prs32_result.iloc[0]['value']}")

result_df


## Query 3: Combined Report (PRS30006032 Q01 + Population)


In [ ]:
# Query 3: Combined Report (PRS30006032 Q01 + Population)
# Use the analytics module function
result_df = query3_combined_report(bls_df, population_df)

print("=" * 60)
print("Query 3: Combined Report (PRS30006032 Q01 + Population)")
print("=" * 60)
print(f"\nTotal records: {len(result_df)}")
print(f"Records with population data: {result_df['Population'].notna().sum()}")

# Show all records with population data
result_with_pop = result_df[result_df['Population'].notna()].sort_values('year')
print(f"\nRecords with Population Data ({len(result_with_pop)} records):")
print(result_with_pop.to_string(index=False))

# Show records for 2013-2018 specifically
result_2013_2018 = result_with_pop[
    (result_with_pop['year'] >= 2013) & (result_with_pop['year'] <= 2018)
]
print(f"\nRecords for 2013-2018 ({len(result_2013_2018)} records):")
print(result_2013_2018.to_string(index=False))

# Show all records (including those without population data)
print(f"\nAll Records (including those without population data):")
result_df.sort_values('year')


## Summary

All three analytical queries have been successfully executed:

1. **Query 1**: Calculated mean and standard deviation of US population (2013-2018)
2. **Query 2**: Found the best year (year with maximum sum of values) for each series_id
3. **Query 3**: Generated combined report for PRS30006032 Q01 with population data

The results show:
- Population statistics for the specified time period
- Best performing year for each time series
- Combined view of economic indicator (PRS30006032 Q01) with population data
